In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 

df = sns.load_dataset("titanic")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.describe()


In [ ]:
df.isnull().sum()

In [ ]:
(df == "").sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns))


In [ ]:
df['embarked'].unique()

In [ ]:
for columns in df:
   print(f"\n{columns}:")
   print(df[columns].unique())

In [ ]:
df["sex"].value_counts()

In [ ]:
for columns in df:
   print(df[columns].value_counts())

In [ ]:
df.describe().columns

In [ ]:
numeric_columns = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
for num_cols in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[num_cols] ,kde = True,bins = 20  )

In [ ]:
for num_cols in numeric_columns:
    plt.figure(figsize=(16, 10))
    sns.boxplot(x = df[num_cols])

In [ ]:
plt.figure(figsize=(24,12))
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
categorical_cols = df.select_dtypes(include = object).columns
categorical_cols

In [ ]:
sns.countplot(x = df['sex'])

In [ ]:
for cols in categorical_cols:
    sns.countplot(x = df[cols])
    plt.show()


Data Cleaning and preprocessing

In [ ]:
df_cleaned = df.copy()
df_cleaned.info()

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.drop_duplicates(inplace = True)

In [ ]:

df_cleaned.shape

In [ ]:
df_cleaned.isnull().sum()



In [ ]:
df_cleaned = df_cleaned.drop(columns=['deck'])
df_cleaned = df_cleaned.drop(columns=['alive'])

In [ ]:
df_cleaned['age'].fillna(df_cleaned['age'].median(), inplace=True)

In [ ]:
df_cleaned.info()

In [ ]:
df_cleaned['embarked'] = df_cleaned['embarked'].fillna(df_cleaned['embarked'].mode()[0])

In [ ]:
df_cleaned['embark_town'] = df_cleaned['embark_town'].fillna(df_cleaned['embark_town'].mode()[0])

In [ ]:
df_cleaned.info()

Handle Incorrect Values

In [ ]:
df_cleaned[df_cleaned['age']<0]

In [ ]:
numeric_columns = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
for num_cols in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(df_cleaned[num_cols])

In [ ]:
# for removal of outlier let use IQR
# let remove the outlier in the fare column

# Calculate Q1 and Q3
Q1 = df_cleaned["fare"].quantile(0.25)
Q3 = df_cleaned["fare"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate lower and upper limits
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR



In [ ]:
# Find outliers
outliers = df_cleaned[(df_cleaned["fare"] < lower_limit) | (df_cleaned["fare"] > upper_limit)]

print("\nNumber of Outliers :", len(outliers))
print(outliers[["fare"]])

In [ ]:
df_cleaned['fare'] = df_cleaned['fare'].clip(lower=lower_limit, upper=upper_limit)
# OR df_cleaned['fare'] = np.where(df_cleaned['fare'] > upper_limit , upper_limit , df_cleaned['fare'] )

In [ ]:
sns.boxplot(df_cleaned['fare'])

In [ ]:
# Q1 = df_cleaned["age"].quantile(0.25)
# Q3 = df_cleaned["age"].quantile(0.75)

# # Calculate IQR
# IQR = Q3 - Q1

# # Calculate lower and upper limits
# lower_limit = Q1 - 1.5 * IQR
# upper_limit = Q3 + 1.5 * IQR

# outliers = df_cleaned[(df_cleaned["age"] < lower_limit) | (df_cleaned["age"] > upper_limit)]

# print("\nNumber of Outliers :", len(outliers))
# print(outliers[["age"]])

In [ ]:
numerical_cols = ['survived', 'pclass', 'sibsp', 'parch', 'fare']

for col in numerical_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    df_cleaned[col] = df_cleaned[col].clip(lower=lower_limit, upper=upper_limit)

In [ ]:
for num_cols in numerical_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(df_cleaned[num_cols])

In [ ]:
df_cleaned.head(15)

In [ ]:
categorical_cols = df_cleaned.select_dtypes(include=[object, bool, "category"])
categorical_cols


In [ ]:
df_cleaned['adult_male'] = df_cleaned['adult_male'].astype(int)
df_cleaned['alone'] = df_cleaned['alone'].astype(int)
df_cleaned['sex'] = df_cleaned['sex'].map({"male" : 0,"female" : 1})
df_cleaned

In [ ]:
from sklearn.preprocessing import LabelEncoder


le = LabelEncoder()


cols_to_encode = ["who", "class", "embarked"]


for col in cols_to_encode:
    df_cleaned[col] = le.fit_transform(df_cleaned[col])

df_cleaned.head(11)

In [ ]:
df['embark_town'].unique()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns = ['embark_town'])

In [ ]:
df_cleaned.head()

In [ ]:
bool_cols = ['embark_town_Cherbourg','embark_town_Queenstown','embark_town_Southampton']

df_cleaned[bool_cols] = df_cleaned[bool_cols].astype(int)

In [ ]:
df_cleaned.columns

In [ ]:
# finding corr 

from scipy.stats import pearsonr

# ----------------------------------
# Pearson Correlation Calculation
# ----------------------------------

# List of features to check against target
selected_features = [
     'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'alone',
       'embark_town_Cherbourg', 'embark_town_Queenstown',
       'embark_town_Southampton'
]

correlations = {
    feature: pearsonr(df_cleaned[feature], df_cleaned['survived'])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

steps = [
    "Collect Data",
    "EDA",
    "Clean Data",
    "Handle Missing Values",
    "Remove Duplicates",
    "Handle Outliers",
    "Encode Categorical Data",
    "Feature Engineering",
    "Feature Selection",
    
]

fig, ax = plt.subplots(figsize=(6, 12))
ax.set_xlim(0, 1)
ax.set_ylim(0, len(steps))
ax.axis("off")

for i, step in enumerate(steps):
    y = len(steps) - i - 0.5

    # Draw box
    ax.text(
        0.5, y, step,
        ha="center", va="center",
        fontsize=11,
        bbox=dict(boxstyle="round", facecolor="lightblue", edgecolor="black")
    )

    # Draw arrow
    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(0.5, y - 0.55),
            xytext=(0.5, y - 0.15),
            arrowprops=dict(arrowstyle="->", lw=2)
        )

plt.title("Machine Learning Workflow", fontsize=14, weight="bold")
plt.show()